# Час 12 - Смањење историје разговора са агенцијом за белешке

Овај бележник демонстрира како управљати контекстом у дугим разговорима користећи Microsoft Agent Framework. Како разговори расту, број токена се повећава — на крају прелази лимит контекстног прозора модела. Ово решавамо уз помоћ **шаблона за сумирање контекста** и **агенцијске свеске** за упорну меморију.

## Шта ћете научити:
1. **Зашто је важно управљање контекстом**: Разумевање ограничења броја токена и контекстних прозора
2. **Агенти свесни контекста**: Прављење агената који управљају сопственим контекстом разговора
3. **Шаблон за сумирање контекста**: Коришћење алата за кондензацију историје разговора
4. **Агенцијска свеска**: Упорна меморија која опстаје након смањења контекста

## Претпоставке:
- Azure OpenAI подешен са конфигурисаним променљивим окружења
- Разумевање основних концепата агената из претходних часова


## Подешавање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Зашто је управљање контекстом важно

Сваки LLM има ограничени **прозор контекста** — максималан број токена које може обрадити у једном захтеву. Како се вишекратни разговор наставља:

- **Број токена расте линеарно** са сваком поруком корисника и одговором асистента.
- **Токени упита доминирају трошковима** јер се цела историја поново шаље сваки пут.
- На крају разговор **прелази прозор контекста** и модел или скраћује или прави грешку.

### Стратегије за управљање контекстом

| Стратегија | Како функционише | Компромис |
|---|---|---|
| **Скраћивање** | Избацују се најстарије поруке | Губи се ранији контекст |
| **Сажимање** | Кондензују се старије поруке у резиме | Неко детаљно се губи, али кључне тачке остају |
| **Бележница / Спољна меморија** | Чувају се кључне чињенице ван разговора | Захтева позиве алата, али преживљава било какво смањење |

У овом бележнику комбинујемо **сажимање** са алатом **бележнице** како би агент могао да одржи континуитет чак и када се историја разговора кондензује.


## Креирање агента свесног контекста


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Симулирање дугог разговора

Хајде да прођемо кроз разговор са више корака да видимо како се контекст акумулира. Агент треба да задржи кључне детаље (преференције, буџет, датуме путовања) кроз кораке и да прикаже континуитет.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Обратите пажњу како агент задржава контекст из претходних корака — он зна о Јапану, сушијуу, храмовима, фотографији, буџету од 3000 долара, самосталном путовању и априлском временском периоду. У кратком разговору ово добро функционише, али како разговор расте, цела историја постаје скупа за поновно слање.

Хајде да наставимо разговор са више корака да видимо акумулацију контекста:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Образац за сумирање контекста

Како разговор расте, можемо користити **алат за сумирање** да саберемо акумулирани контекст у компактном формату. Агент позива овај алат да забележи кључне преференције како би чак и ако се старије поруке уклоне, суштинске информације биле сачуване.

Овај образац је основа за напредније смањење историје:
1. Агент идентификује кључне чињенице из разговора
2. Позива алат за сумирање да их сачува
3. Старије поруке могу безбедно да се уклоне јер резиме обухвата оно што је важно

Испод дефинишемо алат `summarize_preferences` који агент може да позове да забележи компактни резиме онога што је научио.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Резиме

У овој лекцији сте научили како да управљате контекстом у дуготрајним агентским разговорима користећи Microsoft Agent Framework:

### Кључни појмови
- **Контекстуалне прозоре су ограничене** — сваки токен у историји разговора кошта новац и рачуна се у оквир ограничења.
- **Инструменти за резимирање** омогућавају агенту да сабере акумулирани контекст у компактне сажетке, смањујући употребу токена уз задржавање битних информација.
- **Агентске пречице** пружају трајну спољашњу меморију која преживљава свако смањење разговора.

### Шта сте изградили
- **Агент свестан контекста** који одржава континуитет кроз вишекратне дијалоге
- **Инструмент за резимирање** (`summarize_preferences`) који бележи кључне податке корисника у компактном формату
- **Вишекратни дијалог** који демонстрира задржавање контекста и обраду промена

### Примена у стварном свету
- **Ботови за корисничку подршку**: памте преференције кроз дуге сесије подршке
- **Персонални асистенти**: прате текуће пројекте без поновног објашњавања контекста
- **Образовни тутори**: одржавају напредак ученика кроз бројне интеракције

### Следећи кораци
- Имплементирати потпуни алат за пречице са чувањем у фајловима
- Додати аутоматско скраћивање историје након резимирања
- Комбиновати са векторским базама података за семантичко претраживање меморије
- Направити агенте који могу наставити разговоре данима касније са потпуним контекстом


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем AI сервиса за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да превод буде прецизан, молимо вас да имате у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било какве неспоразуме или погрешна тумачења која могу настати употребом овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
